# 🇬🇭 Modelling Fiscal Stress of the Ghanaian Economy
## XGBoost Binary Classification Pipeline | 1990–2023

**Author:** [Your Name]  
**Date:** 2024  
**Algorithm:** XGBoost (eXtreme Gradient Boosting)  
**Task:** Binary classification — Fiscal Stress (1) vs. No Stress (0)

---
### Project Overview
This notebook implements a complete machine learning pipeline to **identify, predict, and interpret fiscal stress episodes** in Ghana using annual macroeconomic panel data (1990–2023). The pipeline includes:

1. 📦 Library installation & imports
2. 📊 Data loading & inspection
3. 🔍 Exploratory Data Analysis (EDA)
4. 🏷️ Target label construction
5. ⚙️ Feature engineering & preprocessing
6. 🤖 XGBoost model training & hyperparameter tuning
7. 📈 Model evaluation (AUC-ROC, F1, Confusion Matrix)
8. 🔎 SHAP explainability & feature importance
9. 📉 Walk-forward cross-validation
10. 🚨 Early Warning Score output


## 1. Install & Import Libraries

In [ ]:
# Install required libraries (run once in Colab)
!pip install xgboost shap imbalanced-learn optuna --quiet


In [ ]:
# ── Core libraries ──────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Machine Learning ─────────────────────────────────────────────────────────
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_score, TimeSeriesSplit
from sklearn.metrics import (roc_auc_score, f1_score, precision_score,
                              recall_score, accuracy_score, confusion_matrix,
                              RocCurveDisplay, classification_report, brier_score_loss)
from sklearn.pipeline import Pipeline
from sklearn.calibration import CalibratedClassifierCV

# ── Imbalanced learning ──────────────────────────────────────────────────────
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

# ── SHAP explainability ──────────────────────────────────────────────────────
import shap

# ── Hyperparameter optimisation ──────────────────────────────────────────────
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── Plot settings ────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 120,
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
})
PALETTE = ['#2E7D32', '#B71C1C']   # green=no stress, red=stress
print("✅ All libraries loaded successfully")


## 2. Load & Inspect the Dataset

In [ ]:
# ── Ghana Fiscal Dataset (1990–2023) ─────────────────────────────────────────
# Upload 'Ghana_Fiscal_Stress_Dataset.xlsx' to Colab or run the cell below
# to recreate the dataset inline.

data_dict = {
    'Year':            [1990,1991,1992,1993,1994,1995,1996,1997,1998,1999,
                        2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,
                        2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,
                        2020,2021,2022,2023],
    'GDP_Growth':      [3.3,5.3,3.9,4.9,3.3,4.1,4.6,4.2,4.7,4.4,
                        3.7,4.0,4.5,5.2,5.6,5.9,6.4,6.5,8.4,4.8,
                        7.9,14.0,9.3,7.3,2.9,2.2,3.6,8.1,6.3,6.5,
                        0.4,5.4,3.2,2.9],
    'Inflation':       [37.3,18.0,10.1,24.9,24.9,59.5,32.7,27.9,14.6,12.4,
                        25.1,32.9,14.8,26.7,12.6,15.1,10.2,10.7,16.5,19.3,
                        10.7,8.7,9.2,11.7,17.0,17.2,17.5,12.4,9.8,7.1,
                        9.9,10.0,31.9,38.1],
    'Debt_to_GDP':     [82.5,78.2,79.5,81.2,82.8,84.1,80.5,78.3,76.2,80.5,
                        89.2,91.5,86.3,78.5,71.2,65.8,42.5,40.2,44.8,52.5,
                        48.5,47.2,55.5,58.8,70.5,73.2,73.8,71.8,62.5,63.5,
                        76.5,80.5,92.8,88.5],
    'Primary_Balance': [-3.2,-2.8,-3.5,-3.8,-2.9,-2.1,-1.8,-1.5,-2.3,-4.2,
                        -5.8,-5.2,-3.8,-2.5,-1.2,-0.8,0.5,0.3,-2.5,-5.8,
                        -3.2,-2.8,-5.8,-7.2,-5.5,-4.8,-3.8,-2.5,-1.8,-2.8,
                        -8.5,-6.2,-10.5,-4.2],
    'Revenue_GDP':     [12.1,13.2,13.8,14.2,15.1,16.2,17.5,18.2,18.9,17.8,
                        16.5,17.2,18.5,19.8,21.5,22.8,24.2,25.5,24.8,22.5,
                        24.8,25.5,24.2,22.8,21.5,22.8,23.5,24.8,26.2,24.8,
                        20.5,19.8,18.5,20.2],
    'Expenditure_GDP': [18.4,17.5,18.9,19.5,20.2,19.8,20.5,21.1,22.5,24.5,
                        25.8,26.8,24.2,24.1,23.8,24.5,25.2,26.8,29.8,32.5,
                        31.5,29.8,33.8,35.5,30.8,31.5,30.5,29.5,28.5,29.5,
                        35.8,29.8,35.8,32.5],
    'External_Debt_GDP':[65.2,62.3,63.8,65.5,66.2,68.5,65.8,63.2,61.5,68.2,
                         72.5,75.2,70.1,63.5,58.2,52.5,35.8,28.5,30.5,35.2,
                         33.8,32.5,35.8,38.5,48.5,52.5,55.2,52.8,48.5,50.2,
                         58.5,62.5,72.5,68.5],
    'Debt_Service_Ratio':[22.8,20.1,19.5,21.3,22.1,23.5,21.8,20.5,19.2,24.3,
                          27.8,26.5,22.3,18.9,15.2,13.8,10.2,8.5,9.8,11.5,
                          10.8,9.5,11.2,13.5,16.8,18.5,19.2,17.5,15.8,16.5,
                          18.8,20.5,25.8,28.5],
    'CA_Balance':      [-6.2,-5.8,-7.2,-6.5,-6.9,-5.8,-4.9,-5.2,-5.8,-7.5,
                        -8.9,-8.2,-6.5,-5.8,-5.2,-5.5,-6.2,-7.8,-11.2,-8.5,
                        -8.8,-9.5,-12.5,-11.8,-9.5,-7.2,-6.8,-4.8,-3.2,-2.8,
                        -3.5,-2.8,-3.5,-2.2],
    'Reserves_Months': [2.1,2.3,2.0,1.8,1.7,1.9,2.2,2.4,2.8,2.1,
                        1.5,1.8,2.3,2.8,3.1,3.5,4.2,4.8,3.9,3.5,
                        3.8,4.2,3.2,2.9,2.5,2.8,3.1,3.5,3.8,4.2,
                        3.2,3.5,2.1,3.8],
    'FX_Depreciation': [15.2,12.3,14.5,25.3,30.1,28.7,12.8,20.5,8.3,37.8,
                        49.8,16.5,15.2,24.8,2.2,1.8,1.5,3.5,20.8,28.5,
                        1.5,5.2,17.5,15.8,31.2,15.5,9.5,4.8,8.5,13.8,
                        4.5,4.1,29.8,19.5],
    'Interest_Revenue':[28.4,25.8,27.3,29.4,27.8,26.5,24.3,23.1,22.5,28.7,
                        31.5,32.8,29.5,25.3,20.1,17.5,14.8,13.2,15.8,18.5,
                        17.8,16.5,22.8,25.3,29.8,32.5,31.8,29.5,26.8,27.5,
                        34.8,37.5,48.5,42.8],
    'IMF_Program':     [1,1,0,0,0,0,0,0,0,0,1,1,1,1,0,0,0,0,0,1,0,0,0,0,1,1,1,1,0,0,0,0,0,1],
    'Fiscal_Stress':   [1,1,1,1,1,1,1,1,1,1,1,1,1,1,0,0,0,0,0,0,0,0,0,1,1,1,1,0,0,0,1,1,1,1],
}

df = pd.DataFrame(data_dict)

print(f"Dataset shape: {df.shape}")
print(f"\nFiscal Stress distribution:")
print(df['Fiscal_Stress'].value_counts().rename({0:'No Stress', 1:'Fiscal Stress'}))
print(f"\nClass ratio — Stress: {df['Fiscal_Stress'].mean():.1%}")
df.head(10)


In [ ]:
# ── Basic statistics ─────────────────────────────────────────────────────────
print("\n📊 DESCRIPTIVE STATISTICS")
print("="*60)
df.describe().round(2)


In [ ]:
# ── Missing values check ─────────────────────────────────────────────────────
print("Missing values per column:")
print(df.isnull().sum())
print("\n✅ No missing values in this dataset")


## 3. Exploratory Data Analysis (EDA)

In [ ]:
# ── 3.1 Fiscal Stress Timeline ──────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(14, 11), sharex=True)
fig.suptitle("Ghana Key Fiscal Indicators & Stress Episodes (1990–2023)",
             fontsize=14, fontweight='bold', y=1.01)

stress_years = df[df['Fiscal_Stress'] == 1]['Year']

def shade_stress(ax):
    for yr in stress_years:
        ax.axvspan(yr - 0.5, yr + 0.5, alpha=0.18, color='#B71C1C', zorder=0)

# Plot 1: Debt & Interest/Revenue
axes[0].plot(df['Year'], df['Debt_to_GDP'],      color='#1B5E20', lw=2, marker='o', ms=4, label='Debt/GDP (%)')
axes[0].plot(df['Year'], df['Interest_Revenue'], color='#F9A825', lw=2, marker='s', ms=4, label='Interest/Revenue (%)')
axes[0].axhline(70,  color='#1B5E20', ls='--', alpha=0.5, lw=1)
axes[0].axhline(30,  color='#F9A825', ls='--', alpha=0.5, lw=1)
shade_stress(axes[0])
axes[0].legend(fontsize=9); axes[0].set_ylabel("Percent (%)")
axes[0].set_title("Debt Sustainability Indicators  (dashed = stress threshold)")

# Plot 2: GDP growth & Inflation
axes[1].plot(df['Year'], df['GDP_Growth'], color='#2E7D32', lw=2, marker='o', ms=4, label='GDP Growth (%)')
axes[1].plot(df['Year'], df['Inflation'],  color='#E53935', lw=2, marker='D', ms=4, label='Inflation (%)')
shade_stress(axes[1])
axes[1].legend(fontsize=9); axes[1].set_ylabel("Percent (%)")
axes[1].set_title("Real Economy & Monetary Conditions")

# Plot 3: Primary balance & FX depreciation
axes[2].bar(df['Year'], df['Primary_Balance'], color=df['Fiscal_Stress'].map({0:'#2E7D32', 1:'#B71C1C'}), alpha=0.75)
shade_stress(axes[2])
axes[2].axhline(0, color='black', lw=1)
axes[2].set_ylabel("% of GDP"); axes[2].set_xlabel("Year")
axes[2].set_title("Primary Balance (% GDP)  — red bars = stress years")

for ax in axes:
    ax.set_xlim(1989, 2024)
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('ghana_fiscal_timeline.png', bbox_inches='tight', dpi=130)
plt.show()
print("\n🔴 Red shading = fiscal stress episodes")


In [ ]:
# ── 3.2 Correlation Heatmap ─────────────────────────────────────────────────
features = ['GDP_Growth','Inflation','Debt_to_GDP','Primary_Balance',
            'Revenue_GDP','Expenditure_GDP','External_Debt_GDP',
            'Debt_Service_Ratio','CA_Balance','Reserves_Months',
            'FX_Depreciation','Interest_Revenue','IMF_Program','Fiscal_Stress']

corr = df[features].corr()

fig, ax = plt.subplots(figsize=(13, 10))
mask = np.zeros_like(corr, dtype=bool)
mask[np.triu_indices_from(mask)] = True

sns.heatmap(corr, mask=mask, cmap='RdYlGn_r', center=0,
            annot=True, fmt='.2f', annot_kws={'size':7.5},
            linewidths=0.4, ax=ax, vmin=-1, vmax=1,
            cbar_kws={'shrink':0.75})
ax.set_title("Pearson Correlation Matrix — Ghana Fiscal Variables (1990–2023)",
             fontsize=12, fontweight='bold', pad=12)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.savefig('correlation_heatmap.png', bbox_inches='tight', dpi=130)
plt.show()

# Top correlates with Fiscal_Stress
print("\n📊 Top correlates with Fiscal Stress:")
print(corr['Fiscal_Stress'].drop('Fiscal_Stress').abs().sort_values(ascending=False).round(3).to_string())


In [ ]:
# ── 3.3 Distribution by Stress Class ────────────────────────────────────────
key_vars = ['Debt_to_GDP','Interest_Revenue','Primary_Balance',
            'GDP_Growth','Inflation','Reserves_Months']
labels   = ['Debt/GDP (%)','Interest/Rev (%)','Primary Balance (%)','GDP Growth (%)','Inflation (%)','Reserves (months)']

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
fig.suptitle("Feature Distributions: Fiscal Stress vs. No Stress",
             fontsize=13, fontweight='bold')

for ax, var, label in zip(axes.flatten(), key_vars, labels):
    for val, color, lbl in [(0,'#2E7D32','No Stress'), (1,'#B71C1C','Fiscal Stress')]:
        subset = df[df['Fiscal_Stress'] == val][var]
        subset.plot.kde(ax=ax, color=color, linewidth=2, label=lbl)
        ax.axvline(subset.mean(), color=color, ls='--', lw=1.2, alpha=0.7)
    ax.set_title(label, fontsize=10, fontweight='bold')
    ax.set_xlabel(''); ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('distribution_by_stress.png', bbox_inches='tight', dpi=130)
plt.show()


## 4. Feature Engineering & Preprocessing

In [ ]:
# ── 4.1 Engineer additional features ────────────────────────────────────────
df_fe = df.copy()

# Lag features (t-1): capture momentum effects
lag_cols = ['Debt_to_GDP','Primary_Balance','GDP_Growth','Inflation',
            'FX_Depreciation','Interest_Revenue']
for col in lag_cols:
    df_fe[f'{col}_lag1'] = df_fe[col].shift(1)
    df_fe[f'{col}_lag2'] = df_fe[col].shift(2)

# Fiscal space indicator
df_fe['Fiscal_Space']     = df_fe['Revenue_GDP'] - df_fe['Interest_Revenue']

# Debt acceleration (year-on-year change in debt ratio)
df_fe['Debt_Acceleration'] = df_fe['Debt_to_GDP'].diff()

# Expenditure gap (spending above revenue)
df_fe['Expenditure_Gap']   = df_fe['Expenditure_GDP'] - df_fe['Revenue_GDP']

# Stress risk composite (rule-based pre-signal)
df_fe['Stress_Risk_Score'] = (
    (df_fe['Debt_to_GDP']    > 65).astype(int) +
    (df_fe['Interest_Revenue']> 25).astype(int) +
    (df_fe['Primary_Balance'] < -3).astype(int) +
    (df_fe['Reserves_Months'] < 2.5).astype(int) +
    (df_fe['FX_Depreciation'] > 15).astype(int)
)

# Drop rows with NaN from lags
df_fe.dropna(inplace=True)
df_fe.reset_index(drop=True, inplace=True)

print(f"Dataset after feature engineering: {df_fe.shape}")
print(f"New features added: {df_fe.shape[1] - df.shape[1]}")
print(f"\nAll features:")
for col in df_fe.columns:
    print(f"  • {col}")


In [ ]:
# ── 4.2 Define X and y ───────────────────────────────────────────────────────
feature_cols = [c for c in df_fe.columns if c not in ['Year','Fiscal_Stress']]

X = df_fe[feature_cols].values
y = df_fe['Fiscal_Stress'].values

print(f"Feature matrix X: {X.shape}")
print(f"Target vector  y: {y.shape}")
print(f"Class balance  — Stress: {y.sum()} | No-Stress: {(y==0).sum()}")
print(f"\nFeature names ({len(feature_cols)}):")
for i, f in enumerate(feature_cols):
    print(f"  {i+1:2d}. {f}")


In [ ]:
# ── 4.3 Standardise features ─────────────────────────────────────────────────
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ── 4.4 SMOTE oversampling to handle class imbalance ────────────────────────
smote   = SMOTE(random_state=42, k_neighbors=3)
X_res, y_res = smote.fit_resample(X_scaled, y)

print(f"After SMOTE — Stress: {y_res.sum()} | No-Stress: {(y_res==0).sum()}")
print(f"Resampled matrix: {X_res.shape}")


## 5. XGBoost Model Training & Hyperparameter Tuning

In [ ]:
# ── 5.1 Baseline XGBoost ─────────────────────────────────────────────────────
ratio = (y == 0).sum() / (y == 1).sum()   # for scale_pos_weight

xgb_base = XGBClassifier(
    n_estimators     = 200,
    max_depth        = 3,
    learning_rate    = 0.05,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    scale_pos_weight = ratio,
    use_label_encoder= False,
    eval_metric      = 'logloss',
    random_state     = 42,
    verbosity        = 0,
)
xgb_base.fit(X_scaled, y)
y_pred_base  = xgb_base.predict(X_scaled)
y_proba_base = xgb_base.predict_proba(X_scaled)[:, 1]

print("Baseline XGBoost (in-sample) — sanity check")
print(f"  AUC-ROC  : {roc_auc_score(y, y_proba_base):.4f}")
print(f"  F1-Score : {f1_score(y, y_pred_base):.4f}")
print(f"  Accuracy : {accuracy_score(y, y_pred_base):.4f}")


In [ ]:
# ── 5.2 Bayesian Hyperparameter Optimisation (Optuna) ───────────────────────
tscv = TimeSeriesSplit(n_splits=5)

def objective(trial):
    params = {
        'n_estimators'    : trial.suggest_int('n_estimators', 50, 400),
        'max_depth'       : trial.suggest_int('max_depth', 2, 6),
        'learning_rate'   : trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample'       : trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha'       : trial.suggest_float('reg_alpha',  0.0, 2.0),
        'reg_lambda'      : trial.suggest_float('reg_lambda', 0.5, 5.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'scale_pos_weight': ratio,
        'use_label_encoder': False,
        'eval_metric'     : 'logloss',
        'random_state'    : 42,
        'verbosity'       : 0,
    }
    model  = XGBClassifier(**params)
    scores = cross_val_score(model, X_scaled, y, cv=tscv,
                             scoring='roc_auc', n_jobs=-1)
    return scores.mean()

study = optuna.create_study(direction='maximize',
                            sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=80, show_progress_bar=False)

print(f"✅ Best AUC-ROC (CV): {study.best_value:.4f}")
print(f"\nBest hyperparameters:")
for k, v in study.best_params.items():
    print(f"  {k:<22} : {v}")


In [ ]:
# ── 5.3 Train final model with best params ──────────────────────────────────
best_params = study.best_params.copy()
best_params.update({
    'scale_pos_weight': ratio,
    'use_label_encoder': False,
    'eval_metric': 'logloss',
    'random_state': 42,
    'verbosity': 0,
})

xgb_final = XGBClassifier(**best_params)
xgb_final.fit(X_scaled, y)

y_pred  = xgb_final.predict(X_scaled)
y_proba = xgb_final.predict_proba(X_scaled)[:, 1]

print("Final XGBoost Model — Training Performance")
print("="*45)
print(classification_report(y, y_pred, target_names=['No Stress','Fiscal Stress']))


## 6. Model Evaluation & Comparison

In [ ]:
# ── 6.1 Walk-Forward Cross-Validation ──────────────────────────────────────
print("🔄 Walk-Forward (Time-Series) Cross-Validation")
print("="*50)

models = {
    'XGBoost (Tuned)' : xgb_final,
    'Random Forest'   : RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42),
    'Logistic Reg.'   : LogisticRegression(class_weight='balanced', max_iter=500, random_state=42),
    'SVM (RBF)'       : CalibratedClassifierCV(SVC(kernel='rbf', class_weight='balanced', random_state=42)),
}

results = {}
for name, model in models.items():
    auc  = cross_val_score(model, X_scaled, y, cv=tscv, scoring='roc_auc').mean()
    f1   = cross_val_score(model, X_scaled, y, cv=tscv, scoring='f1').mean()
    rec  = cross_val_score(model, X_scaled, y, cv=tscv, scoring='recall').mean()
    prec = cross_val_score(model, X_scaled, y, cv=tscv, scoring='precision').mean()
    results[name] = {'AUC-ROC': auc, 'F1-Score': f1, 'Recall': rec, 'Precision': prec}
    print(f"  {name:<22}  AUC={auc:.3f}  F1={f1:.3f}  Recall={rec:.3f}  Prec={prec:.3f}")

results_df = pd.DataFrame(results).T.round(3)
print("\n", results_df.to_string())


In [ ]:
# ── 6.2 Confusion Matrix & ROC Curve ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Confusion matrix
cm = confusion_matrix(y, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=['No Stress','Fiscal Stress'],
            yticklabels=['No Stress','Fiscal Stress'],
            ax=axes[0], linewidths=1)
axes[0].set_title('Confusion Matrix — XGBoost (Final Model)', fontweight='bold')
axes[0].set_ylabel('True Label'); axes[0].set_xlabel('Predicted Label')

# ROC Curve
RocCurveDisplay.from_predictions(y, y_proba, ax=axes[1],
    color='#1B5E20', name=f'XGBoost (AUC = {roc_auc_score(y, y_proba):.3f})')
axes[1].plot([0,1],[0,1], 'k--', lw=1, label='Random Classifier')
axes[1].set_title('ROC Curve — XGBoost Fiscal Stress Classifier', fontweight='bold')
axes[1].legend(fontsize=10)

plt.tight_layout()
plt.savefig('model_evaluation.png', bbox_inches='tight', dpi=130)
plt.show()

# Print key metrics
print("\n📈 Final Model — Key Metrics (full training set)")
print(f"  AUC-ROC   : {roc_auc_score(y, y_proba):.4f}")
print(f"  F1-Score  : {f1_score(y, y_pred):.4f}")
print(f"  Recall    : {recall_score(y, y_pred):.4f}")
print(f"  Precision : {precision_score(y, y_pred):.4f}")
print(f"  Accuracy  : {accuracy_score(y, y_pred):.4f}")
print(f"  Brier     : {brier_score_loss(y, y_proba):.4f}")


In [ ]:
# ── 6.3 Model Comparison Bar Chart ──────────────────────────────────────────
metrics_list = ['AUC-ROC','F1-Score','Recall','Precision']
x = np.arange(len(metrics_list))
width = 0.2
colors = ['#1B5E20','#2196F3','#FF8F00','#9C27B0']

fig, ax = plt.subplots(figsize=(12, 6))
for i, (name, row) in enumerate(results_df.iterrows()):
    vals = [row[m] for m in metrics_list]
    bars = ax.bar(x + i*width, vals, width, label=name, color=colors[i], alpha=0.85)

ax.set_xticks(x + width*1.5)
ax.set_xticklabels(metrics_list, fontsize=11)
ax.set_ylim(0, 1.1); ax.set_ylabel('Score', fontsize=11)
ax.set_title('Model Comparison — Walk-Forward Cross-Validation (5-Fold)',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=9, loc='lower right')
ax.axhline(0.85, color='grey', ls=':', lw=1, label='Target AUC-ROC')
plt.tight_layout()
plt.savefig('model_comparison.png', bbox_inches='tight', dpi=130)
plt.show()


## 7. SHAP Explainability & Feature Importance

In [ ]:
# ── 7.1 SHAP Summary Plot ────────────────────────────────────────────────────
explainer   = shap.TreeExplainer(xgb_final)
shap_values = explainer.shap_values(X_scaled)

shap_df = pd.DataFrame(np.abs(shap_values), columns=feature_cols)
mean_shap = shap_df.mean().sort_values(ascending=False)

print("📊 Mean |SHAP Value| — Feature Importance Ranking")
print("="*50)
for i, (feat, val) in enumerate(mean_shap.items(), 1):
    bar = '█' * int(val * 30)
    print(f"  {i:2d}. {feat:<32} {val:.4f}  {bar}")


In [ ]:
# ── 7.2 SHAP Beeswarm Plot ──────────────────────────────────────────────────
plt.figure(figsize=(11, 8))
shap.summary_plot(shap_values, X_scaled, feature_names=feature_cols,
                  show=False, max_display=15,
                  color_bar_label='Feature Value (Low → High)')
plt.title('SHAP Summary Plot — XGBoost Fiscal Stress Model',
          fontsize=12, fontweight='bold', pad=10)
plt.tight_layout()
plt.savefig('shap_summary.png', bbox_inches='tight', dpi=130)
plt.show()
print("\n💡 Red dots = high feature value pushes prediction toward stress")
print("   Blue dots = low feature value pushes prediction toward no-stress")


In [ ]:
# ── 7.3 SHAP Bar Chart ──────────────────────────────────────────────────────
top_n = 12
top_feats = mean_shap.head(top_n)

fig, ax = plt.subplots(figsize=(10, 6))
colors_bar = ['#B71C1C' if i < 3 else '#F9A825' if i < 6 else '#2E7D32'
              for i in range(top_n)]
bars = ax.barh(range(top_n), top_feats.values[::-1], color=colors_bar[::-1], alpha=0.85)
ax.set_yticks(range(top_n))
ax.set_yticklabels(top_feats.index[::-1], fontsize=10)
ax.set_xlabel('Mean |SHAP Value|', fontsize=11)
ax.set_title('Top-12 Feature Importance (SHAP) — Ghana Fiscal Stress Model',
             fontsize=12, fontweight='bold')
for bar, val in zip(bars, top_feats.values[::-1]):
    ax.text(val + 0.001, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=9)
red_p   = mpatches.Patch(color='#B71C1C', label='High importance')
gold_p  = mpatches.Patch(color='#F9A825', label='Medium importance')
green_p = mpatches.Patch(color='#2E7D32', label='Lower importance')
ax.legend(handles=[red_p, gold_p, green_p], fontsize=9)
plt.tight_layout()
plt.savefig('shap_bar.png', bbox_inches='tight', dpi=130)
plt.show()


In [ ]:
# ── 7.4 SHAP Force Plot — High Stress Year (2022) ───────────────────────────
# Find index of year 2022 in the engineered dataset
year_2022_idx = df_fe[df_fe['Year'] == 2022].index[0]

print(f"🔎 SHAP Force Plot — Year 2022 (Fiscal Stress = {y[year_2022_idx]})")
print(f"   Model probability of stress: {y_proba[year_2022_idx]:.4f}")
print()

shap.initjs()
force = shap.force_plot(
    explainer.expected_value,
    shap_values[year_2022_idx],
    X_scaled[year_2022_idx],
    feature_names=feature_cols,
    matplotlib=True,
    show=False,
    figsize=(18, 3),
)
plt.title("SHAP Force Plot — Ghana 2022 (Debt Crisis Year)", fontsize=11, pad=30)
plt.tight_layout()
plt.savefig('shap_force_2022.png', bbox_inches='tight', dpi=130)
plt.show()


## 8. Early Warning Score Output

In [ ]:
# ── 8.1 Generate EWS probabilities for all years ────────────────────────────
df_ews = df_fe[['Year','Fiscal_Stress']].copy()
df_ews['Stress_Probability'] = y_proba
df_ews['Predicted_Stress']   = y_pred
df_ews['Alert_Level'] = pd.cut(
    df_ews['Stress_Probability'],
    bins=[0, 0.35, 0.60, 0.80, 1.0],
    labels=['🟢 Low', '🟡 Moderate', '🟠 High', '🔴 Critical']
)

print("📊 Ghana Fiscal Stress — Early Warning Score (1992–2023)")
print("="*65)
print(f"{'Year':>6}  {'Actual':>8}  {'Prob':>8}  {'Predicted':>10}  {'Alert Level':>12}")
print("-"*65)
for _, row in df_ews.iterrows():
    actual = "STRESS" if row['Fiscal_Stress'] else "No Stress"
    pred   = "STRESS" if row['Predicted_Stress'] else "No Stress"
    print(f"{int(row['Year']):>6}  {actual:>10}  {row['Stress_Probability']:>8.3f}  {pred:>10}  {str(row['Alert_Level']):>14}")


In [ ]:
# ── 8.2 EWS Probability Timeline ────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 6))

ax.fill_between(df_ews['Year'], df_ews['Stress_Probability'],
                where=df_ews['Stress_Probability'] >= 0.60,
                alpha=0.3, color='#B71C1C', label='High/Critical zone')
ax.fill_between(df_ews['Year'], df_ews['Stress_Probability'],
                where=df_ews['Stress_Probability'] < 0.60,
                alpha=0.3, color='#2E7D32', label='Low/Moderate zone')

ax.plot(df_ews['Year'], df_ews['Stress_Probability'],
        color='#1B5E20', lw=2, marker='o', ms=5, label='Stress Probability')

ax.axhline(0.60, color='#E53935', ls='--', lw=1.5, label='Alert threshold (0.60)')
ax.axhline(0.80, color='#B71C1C', ls='--', lw=1.5, label='Critical threshold (0.80)')
ax.axhline(0.35, color='#2E7D32', ls=':',  lw=1.2, label='Low-risk threshold (0.35)')

# Mark actual stress years
for yr in df_ews[df_ews['Fiscal_Stress']==1]['Year']:
    ax.axvspan(yr-0.4, yr+0.4, alpha=0.08, color='#B71C1C')

ax.set_xlim(df_ews['Year'].min()-0.5, df_ews['Year'].max()+0.5)
ax.set_ylim(-0.05, 1.05)
ax.set_xlabel('Year', fontsize=11); ax.set_ylabel('Fiscal Stress Probability', fontsize=11)
ax.set_title('Ghana Fiscal Stress Early Warning Score — XGBoost (1992–2023)',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=9, loc='upper left', ncol=2)
ax.set_xticks(df_ews['Year'][::2]); ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('ews_timeline.png', bbox_inches='tight', dpi=130)
plt.show()
print("\n🚨 Values above 0.60 = trigger MoF/Bank of Ghana fiscal stress review")


## 9. Summary & Policy Recommendations

In [ ]:
# ── Final Summary Report ─────────────────────────────────────────────────────
print("=" * 65)
print("  GHANA FISCAL STRESS MODEL — SUMMARY REPORT")
print("=" * 65)
print(f"\n  DATASET")
print(f"    Period          : 1990–2023 (annual)")
print(f"    Total obs.      : {len(df_fe)} (after lag engineering)")
print(f"    Stress episodes : {y.sum()} years  |  Non-stress: {(y==0).sum()} years")
print(f"    Features used   : {len(feature_cols)}")
print(f"\n  ALGORITHM")
print(f"    Primary model   : XGBoost (eXtreme Gradient Boosting)")
print(f"    Hyperparameter  : Bayesian Optimisation (Optuna, 80 trials)")
print(f"    Validation      : Walk-forward 5-fold CV (time-series aware)")
print(f"\n  PERFORMANCE METRICS (CV)")
cv_auc = cross_val_score(xgb_final, X_scaled, y, cv=tscv, scoring='roc_auc').mean()
cv_f1  = cross_val_score(xgb_final, X_scaled, y, cv=tscv, scoring='f1').mean()
cv_rec = cross_val_score(xgb_final, X_scaled, y, cv=tscv, scoring='recall').mean()
print(f"    AUC-ROC         : {cv_auc:.4f}")
print(f"    F1-Score        : {cv_f1:.4f}")
print(f"    Recall          : {cv_rec:.4f}")
print(f"\n  TOP-3 SHAP DRIVERS")
for i, (feat, val) in enumerate(mean_shap.head(3).items(), 1):
    print(f"    {i}. {feat:<32} SHAP={val:.4f}")
print(f"\n  POLICY RECOMMENDATIONS")
recs = [
    "Maintain Debt/GDP below 65% — primary stress trigger",
    "Reduce Interest/Revenue ratio below 25% via revenue widening",
    "Build foreign reserves to ≥ 4 months import cover (buffer)",
    "Deploy countercyclical policy when GDP growth < 3%",
    "Activate EWS alert protocol when stress score > 0.60",
]
for i, r in enumerate(recs, 1):
    print(f"    {i}. {r}")
print("\n" + "=" * 65)


---
## 📚 Data Sources
- **IMF World Economic Outlook (WEO)** — GDP growth, inflation, debt/GDP, fiscal balance
- **World Bank World Development Indicators (WDI)** — Revenue, expenditure, external debt
- **Bank of Ghana (BoG)** — Exchange rate, reserves, interest rates
- **Ghana Ministry of Finance (MoF)** — Debt sustainability analyses

## 📖 Key References
- Cerovic, S. et al. (2018). *A Fiscal Risk Early-Warning Model*. IMF Working Paper.
- Savona, R. & Vezzoli, M. (2015). *Fitting and Forecasting Sovereign Defaults using Multiple Risk Signals*. Oxford Bulletin of Economics and Statistics.
- Acosta-Ormaechea, S. (2018). *Forecasting Fiscal Crises*. IMF Working Paper.

## ⚖️ Licence
For academic and research use. Cite data sources when publishing results.
